In [ ]:
# Parameters (Papermill will override this)
RESULT_FOLDER_NAME = None

In [ ]:
# Set parent as root and import config
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent))
import configs.simulation_config as cfg
from configs.paths import DATA_DIR, PROCESSED_DIR

import pandas as pd
import numpy as np
from pathlib import Path
from joblib import Parallel, delayed
import pickle

Load result and other data

In [ ]:
# fallback to latest result for manual run
# or set RESULT_FOLDER_NAME to a desired folder
if RESULT_FOLDER_NAME is None:
    result_dir = DATA_DIR / "result"
    RESULT_FOLDER_NAME = sorted([p.name for p in result_dir.iterdir() if p.is_dir()])[-1]

folder_path = Path(f"{DATA_DIR}/result/{RESULT_FOLDER_NAME}")

with open(folder_path / "flow_result.pkl", "rb") as f:
    flow_result = pickle.load(f)

In [ ]:
def build_node_lookups(nodes):
    mode_lookup = nodes.groupby("osmid")["transport_mode"].first()
    pop_lookup = nodes.groupby("osmid")["pop_total"].sum().fillna(0)
    capacity_lookup = nodes.groupby("osmid")["shelter_capacity"].max().fillna(1e6)
    vehicle_occ_lookup = nodes.groupby("osmid")["vehicle_occ"].first().fillna(1)
    evac_dest_lookup = nodes.set_index("osmid")["evac_dest"].to_dict()

    return {
        "transport_mode": mode_lookup,
        "population": pop_lookup,
        "shelter_capacity": capacity_lookup,
        "vehicle_occ": vehicle_occ_lookup,
        "evac_dest": evac_dest_lookup,
    }

In [ ]:
nodes = pd.read_csv(PROCESSED_DIR / "hatyai_nodes_transport.csv")
edges = pd.read_csv(PROCESSED_DIR / "hatyai_edges_with_dynamic_flood.csv")

lookups = build_node_lookups(nodes)

timeline_df = pd.read_parquet(PROCESSED_DIR / "flood_simulation_timeline.parquet")
time_names = 't' + timeline_df['step_id'].astype(str)
time_stamps = timeline_df["timestamp"]
timestep_hour = [
    (time_stamps[i + 1] - time_stamps[i]).total_seconds() / 3600.0
    for i in range(len(time_stamps) - 1)
]

# Flood Depth Visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
import matplotlib.colors as mcolors
import geopandas as gpd
import osmnx as ox

edges_with_flood_df = pd.read_csv(PROCESSED_DIR / "hatyai_edges_with_dynamic_flood.csv")

In [ ]:


depth_cols = [c for c in edges_with_flood_df.columns if c.startswith("flood_depth_t")]
depth_matrix = edges_with_flood_df[depth_cols].values  # (n_edges, n_timesteps)

# ── 1. Time-series of aggregate stats ──────────────────────────────────────
mean_depth = depth_matrix.mean(axis=0)
max_depth  = depth_matrix.max(axis=0)
frac_flooded = (depth_matrix > 0).mean(axis=0)  # fraction of edges with any flood

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

axes[0].plot(time_stamps, mean_depth, color="steelblue")
axes[0].set_ylabel("Mean depth (m)")
axes[0].set_title("Average flood depth across all edges")

axes[1].plot(time_stamps, max_depth, color="crimson")
axes[1].set_ylabel("Max depth (m)")
axes[1].set_title("Maximum flood depth across all edges")

axes[2].plot(time_stamps, frac_flooded * 100, color="darkorange")
axes[2].set_ylabel("% edges flooded")
axes[2].set_title("Fraction of edges with flood depth > 0")
axes[2].set_xlabel("Time")

for ax in axes:
    for ts in time_stamps:
        ax.axvline(ts, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)

plt.tight_layout()
plt.savefig(folder_path / "flood_timeseries.png", dpi=150)
plt.show()

In [ ]:
# ── 2. Spatial snapshots at key timesteps ──────────────────────────────────
# Pick a few representative steps: start, ~25%, ~50%, ~75%, end
n_steps = depth_matrix.shape[1]
snap_indices = [0, n_steps // 4, n_steps // 2, 3 * n_steps // 4, n_steps - 1]

edges_geo = gpd.GeoDataFrame(
    edges_with_flood_df, 
    geometry=gpd.GeoSeries.from_wkt(edges_with_flood_df["geometry"]),
    crs="EPSG:4326"
)

vmax = depth_matrix.max()
norm = mcolors.Normalize(vmin=0, vmax=vmax if vmax > 0 else 1)
cmap = matplotlib.colormaps["YlOrRd"]

fig, axes = plt.subplots(1, len(snap_indices), figsize=(20, 5))

for ax, ti in zip(axes, snap_indices):
    col = f"flood_depth_t{ti}"
    depths = edges_geo[col] if col in edges_geo.columns else pd.Series(0.0, index=edges_geo.index)

    # Dry edges in light gray, flooded edges colored by depth
    dry = edges_geo[depths <= 0]
    wet = edges_geo[depths > 0]

    dry.plot(ax=ax, color="lightgray", linewidth=0.4, aspect=None)
    if not wet.empty:
        wet.plot(ax=ax, column=col, cmap=cmap, norm=norm, linewidth=1.2, aspect=None)

    ax.set_aspect("equal")
    ts_label = time_stamps[ti].strftime("%Y-%m-%d %H:%M")
    ax.set_title(ts_label, fontsize=9)
    ax.set_axis_off()

sm = matplotlib.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
fig.colorbar(sm, ax=axes, orientation="vertical", fraction=0.015, pad=0.02, label="Flood depth (m)")
plt.suptitle("Flood depth spatial distribution", y=1.02)
plt.savefig(folder_path / "flood_spatial_snapshots.png", dpi=150, bbox_inches="tight")
plt.show()

# Evacuation Visualization

In [ ]:
# Evacuee arrivals per time step (people + cumulative success rate)
from collections import defaultdict, deque
import matplotlib.pyplot as plt

flow_dict = flow_result["flow_dict"]
super_source = "super_source"
super_sink = "super_sink"
total_people = float(nodes.get("pop_total", pd.Series(dtype=float)).sum())
L = len(time_names)
eps = 1e-9

# Map exit/shelter collector names back to osmids
out_city_osmids_cell = flow_result.get("out_city_osmids", set())
chosen_shelter_osmids_cell = set(int(o) for o in nodes.loc[nodes["is_shelter"] == True, "osmid"].drop_duplicates())
collector_to_exit = {f"exit_{osmid}": osmid for osmid in out_city_osmids_cell}
collector_to_shelter = {f"shelter_{osmid}": osmid for osmid in chosen_shelter_osmids_cell}

# Residual flow graph for path reconstruction
residual_flow = {(u, v): float(f) for u, nbrs in flow_dict.items() for v, f in nbrs.items() if f > 0}
adj = defaultdict(list)
for (u, v), f in residual_flow.items():
    adj[u].append(v)

def find_path(start, target):
    q = deque([start])
    parent = {start: None}
    while q:
        u = q.popleft()
        if u == target:
            break
        for v in adj.get(u, []):
            if v in parent:
                continue
            if residual_flow.get((u, v), 0.0) <= 0:
                continue
            parent[v] = u
            q.append(v)
    if target not in parent:
        return None
    path = []
    v = target
    while parent[v] is not None:
        u = parent[v]
        path.append((u, v))
        v = u
    path.reverse()
    return path

arrived_people = np.zeros(L)
arrived_units = np.zeros(L)

for osmid in lookups["population"].index:
    supply = flow_dict.get(super_source, {}).get((int(osmid), 0), 0.0)
    if supply <= eps:
        continue
    origin = (int(osmid), 0)
    remaining = supply
    while remaining > eps:
        path = find_path(origin, super_sink)
        if not path:
            break
        bottleneck = min(residual_flow[e] for e in path)
        send = min(bottleneck, remaining)
        for e in path:
            residual_flow[e] -= send
        remaining -= send
        last_u, last_v = path[-1]
        if last_v != super_sink:
            continue

        # Determine arrival layer:
        # - Direct shelter path: last_u is tuple (osmid, layer) → use layer directly
        # - Collector path (shelter/exit): last_u is "shelter_collector_X" or "exit_collector_X"
        #   → look at second-to-last edge for (osmid, layer)
        layer = None
        if isinstance(last_u, tuple):
            layer = int(last_u[1])
        elif isinstance(last_u, str) and (last_u in collector_to_exit or last_u in collector_to_shelter) and len(path) >= 2:
            second_last_u = path[-2][0]
            if isinstance(second_last_u, tuple):
                layer = int(second_last_u[1])

        if layer is not None and 0 <= layer < L:
            arrived_units[layer] += send
            arrived_people[layer] += send

cumulative_people = np.cumsum(arrived_people)
# Round to integers first, then compute rate from the rounded values
arrived_people_int = np.round(arrived_people).astype(int)
cumulative_people_int = np.round(cumulative_people).astype(int)
cumulative_rate = (cumulative_people_int / total_people * 100) if total_people > 0 else np.zeros_like(cumulative_people)
arrival_ts = [time_stamps[i] for i in range(L)]

arrivals_df = pd.DataFrame({
    "time_step": time_names,
    "timestamp": arrival_ts,
    "arrived_people": arrived_people_int,
    "cumulative_people": cumulative_people_int,
    "cumulative_rate_pct": np.round(cumulative_rate, 2),
})
print(arrivals_df)
print(f"\nTotal arrived (sum): {arrived_people.sum():.0f} people")
print(f"Max-flow evacuated:  {flow_result['flow_value']:.0f} people")

fig, ax1 = plt.subplots(figsize=(15, 15))
ax1.bar(range(L), arrived_people, color="#4caf50", alpha=0.7, label="Arrived this step")
ax1.set_ylabel("People arrived")
ax1.set_xlabel("Time step")
ax1.set_xticks(range(L))
ax1.set_xticklabels(time_names, rotation=45, ha="right")

ax2 = ax1.twinx()
ax2.plot(range(L), cumulative_rate, color="#1565c0", marker="o", label="Cumulative evac rate (%)")
ax2.set_ylabel("Cumulative evac rate (%)")
ax2.set_ylim(0, 105)

completion_layer = flow_result.get("evac_completion_layer")
if completion_layer is not None and 0 <= completion_layer < L:
    ax1.axvline(completion_layer, color="#ff9800", linestyle="--", linewidth=1.1, label="Completion layer")

handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(handles1 + handles2, labels1 + labels2, loc="upper left")
ax1.set_title("Evacuee success per time step")
ax1.grid(axis="y", alpha=0.3)
fig.tight_layout()

if "folder_path" in globals():
    csv_path = f"{folder_path}/arrivals_per_time_step.csv"
    arrivals_df.to_csv(csv_path, index=False)
    print(f"Saved arrivals per time step to {csv_path}")

In [ ]:
# Stranded evacuees per time step + mean water level
import matplotlib.pyplot as plt
import numpy as np
import networkx as nx

L = len(time_names)

# ── Stranded people per time step from max-flow ──
# Use arrivals from Cell 22 (arrived_people): stranded = total − cumulative arrived
# This is consistent with Cell 18's residual_pop
cumulative_arrived = np.cumsum(arrived_people)
stranded_people = total_people - cumulative_arrived
stranded_people = np.clip(stranded_people, 0, None)

cumulative_stranded = np.round(stranded_people).astype(int)
stranded_rate_pct = np.round(cumulative_stranded / total_people * 100, 2) if total_people > 0 else np.zeros(L)

# ── Mean flood level per time step ──
mean_flood_depth_all = []
mean_flood_depth_pos = []
for name in time_names:
    col = f"flood_depth_{name}"
    if col in nodes.columns:
        levels = nodes[col].fillna(0)
    elif col in edges.columns:
        levels = edges[col].fillna(0)
    else:
        mean_flood_depth_all.append(0)
        mean_flood_depth_pos.append(0)
        continue
    mean_flood_depth_all.append(float(levels.mean()))
    pos = levels[levels > 0]
    mean_flood_depth_pos.append(float(pos.mean()) if len(pos) > 0 else 0)

# ── Closed edge percentage per time step ──
closed_edge_pct = []
for name in time_names:
    col = f"flood_depth_{name}"
    if col in edges.columns:
        pct = float((edges[col].fillna(0) >= cfg.IMPASSABLE_FLOOD_DEPTH).mean() * 100)
    else:
        pct = 0.0
    closed_edge_pct.append(pct)

# ── Plot: dual-axis ──
fig, ax1 = plt.subplots(figsize=(15, 8))

ax1.bar(range(L), cumulative_stranded, color="#e53935", alpha=0.7, label="Cumulative stranded (people)")
ax1.set_ylabel("Stranded people (cumulative)", color="#e53935", fontsize=12)
ax1.set_xlabel("Time step", fontsize=12)
ax1.set_xticks(range(L))
ax1.set_xticklabels(time_names, rotation=45, ha="right")
ax1.tick_params(axis="y", labelcolor="#e53935")

ax2 = ax1.twinx()
ax2.plot(range(L), mean_flood_depth_all, color="#1565c0", marker="o", linewidth=2,
         label="Mean flood level (all nodes)")
ax2.plot(range(L), mean_flood_depth_pos, color="#0288d1", marker="s", linewidth=1.5,
         linestyle="--", alpha=0.8, label="Mean flood level (flooded nodes)")
ax2.set_ylabel("Mean flood level", color="#1565c0", fontsize=12)
ax2.tick_params(axis="y", labelcolor="#1565c0")

# Third axis: closed edge %
ax3 = ax1.twinx()
ax3.spines["right"].set_position(("outward", 60))
ax3.plot(range(L), closed_edge_pct, color="#ff9800", marker="^", linewidth=1.5,
         linestyle=":", alpha=0.8, label="Closed edges (%)")
ax3.set_ylabel("Closed edges (%)", color="#ff9800", fontsize=12)
ax3.tick_params(axis="y", labelcolor="#ff9800")

# Completion layer marker
completion_layer = flow_result.get("evac_completion_layer")
if completion_layer is not None and 0 <= completion_layer < L:
    ax1.axvline(completion_layer, color="#7b1fa2", linestyle="--", linewidth=1.2, label="Evac completion")

# Combined legend
handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
handles3, labels3 = ax3.get_legend_handles_labels()
ax1.legend(handles1 + handles2 + handles3, labels1 + labels2 + labels3, loc="upper left", fontsize=9)

ax1.set_title("Stranded evacuees per time step + water level", fontsize=14)
ax1.grid(axis="y", alpha=0.3)
fig.tight_layout()

# Save
stranded_df = pd.DataFrame({
    "time_step": time_names,
    "timestamp": [time_stamps[i] for i in range(L)],
    "cumulative_stranded": cumulative_stranded,
    "stranded_rate_pct": stranded_rate_pct,
    "mean_flood_depth": mean_flood_depth_all,
    "mean_flood_depth_flooded": mean_flood_depth_pos,
    "closed_edge_pct": closed_edge_pct,
})
print(stranded_df)

if "folder_path" in globals():
    csv_path = f"{folder_path}/stranded_per_time_step.csv"
    stranded_df.to_csv(csv_path, index=False)
    print(f"Saved stranded per time step to {csv_path}")

In [ ]:
# Animation: network availability and evac movement (red dots flowing from sources to sinks)
from matplotlib.animation import FuncAnimation
from collections import deque, defaultdict as _ddict
import geopandas as gpd
import networkx as nx

graph_filtered = ox.load_graphml(PROCESSED_DIR / "hatyai_graph_clean.graphml")

# Rebuild geometries for animation
edges_anim = (
    ox.graph_to_gdfs(graph_filtered, nodes=False, edges=True)
    .reset_index()[["u", "v", "key", "geometry"]]
    .merge(
        edges[["u", "v", "key"] + [f"flood_depth_{n}" for n in time_names]],
        on=["u", "v", "key"],
        how="left",
    )
)

nodes_anim = gpd.GeoDataFrame(
    nodes,
    geometry=gpd.points_from_xy(nodes.x, nodes.y),
    crs="EPSG:4326",
)
if "residual_pop" not in nodes_anim.columns:
    nodes_anim["pop_sent"] = 0
    nodes_anim["residual_pop"] = nodes_anim.get("pop_total", 0)

out_city_anim = (
    nodes_anim[nodes_anim["is_out_of_city"] == True].copy()
    if "is_out_of_city" in nodes_anim.columns and nodes_anim["is_out_of_city"].any()
    else gpd.GeoDataFrame(columns=nodes_anim.columns)
)

has_dest = "evac_dest" in nodes_anim.columns
pop_series = nodes_anim.get("pop_total", pd.Series(1, index=nodes_anim.index))
pop_norm = (pop_series - pop_series.min()).clip(lower=1)
pop_size = 35 * pop_norm / pop_norm.max()

# Precompute edge geometry lookup
geom_lookup = {}
for _, row in edges_anim.iterrows():
    geom_lookup[(int(row.u), int(row.v))] = row.geometry

flow_dict = flow_result["flow_dict"]
H_flow = flow_result["graph"]
out_city_osmids_anim = flow_result.get("out_city_osmids", set())

# ── KEY FIX ──────────────────────────────────────────────────────────────────
# Reverse-BFS through actual flow edges to find every time-expanded node whose
# flow ultimately reaches an out-of-city exit.  Only checking the immediate
# next-hop (v[0]) misses all intermediate road segments.
rev_adj_flow = _ddict(list)
for u, nbrs in flow_dict.items():
    for v, f in nbrs.items():
        if f > 0 and isinstance(u, tuple) and isinstance(v, tuple):
            rev_adj_flow[v].append(u)

exit_reachable_te = set()
bfs_q = deque()
for osmid in out_city_osmids_anim:
    for layer in range(len(time_names)):
        node = (int(osmid), layer)
        if node not in exit_reachable_te:
            exit_reachable_te.add(node)
            bfs_q.append(node)

while bfs_q:
    node = bfs_q.popleft()
    for prev in rev_adj_flow.get(node, []):
        if prev not in exit_reachable_te:
            exit_reachable_te.add(prev)
            bfs_q.append(prev)

print(f"Time-expanded nodes on exit-bound flow paths: {len(exit_reachable_te)}")
# ─────────────────────────────────────────────────────────────────────────────

# ── Frame-wise reachability (when each origin first loses all paths to any sink) ─
L = len(time_names)
sink_osmids = (
    nodes.loc[
        (nodes.get("is_shelter", False) == True)
        | (nodes.get("is_out_of_city", False) == True),
        "osmid",
    ]
    .drop_duplicates()
    .astype(int)
    .tolist()
)

# Build a lightweight undirected skeleton from the filtered graph for per-frame reachability
G_base = nx.Graph()
G_base.add_nodes_from(int(n) for n in graph_filtered.nodes())
# We will add edges per frame from flood-filtered edge list

reachable_by_frame = []
for ti, name in enumerate(time_names):
    col = f"flood_depth_{name}"
    open_edges = edges.loc[edges[col].fillna(0) < cfg.IMPASSABLE_FLOOD_DEPTH, ["u", "v"]]
    Gi = G_base.copy()
    Gi.add_edges_from((int(u), int(v)) for u, v in zip(open_edges.u, open_edges.v))
    sink_nodes = [s for s in sink_osmids if Gi.has_node(s)]
    if not sink_nodes:
        reachable_by_frame.append(set())
        continue
    reachable = set()
    for s in sink_nodes:
        reachable.update(nx.node_connected_component(Gi, s))
    reachable_by_frame.append(reachable)

stranded_from_layer = {}
origins = nodes.loc[nodes.get("pop_total", 0) > 0, "osmid"].drop_duplicates().astype(int)
for osm in origins:
    first_reach = [ti for ti in range(L) if osm in reachable_by_frame[ti]]
    if not first_reach:
        stranded_from_layer[osm] = 0
        continue
    last_reach = max(first_reach)
    if last_reach < L - 1:
        stranded_from_layer[osm] = last_reach + 1

# Also mark nodes with residual_pop > 0 that are NOT in stranded_from_layer
# (max-flow couldn't evacuate them due to capacity/direction constraints,
#  even though they're still "reachable" in the undirected graph)
evac_completion = flow_result.get("evac_completion_layer", 0) or 0
for _, row in nodes_anim.iterrows():
    osm = int(row["osmid"])
    if row.get("residual_pop", 0) > 0 and osm not in stranded_from_layer:
        stranded_from_layer[osm] = evac_completion

stranded_from_series = nodes_anim["osmid"].map(stranded_from_layer)

# Diagnostics
stranded_t0 = [osm for osm, lay in stranded_from_layer.items() if lay == 0]
stranded_late = [osm for osm, lay in stranded_from_layer.items() if lay and lay > 0]
print(
    f"Stranded timing for {len(stranded_from_layer)} origins: "
    f"t0={len(stranded_t0)}, later={len(stranded_late)}, "
    f"(includes {sum(1 for _, r in nodes_anim.iterrows() if r.get('residual_pop',0) > 0)} nodes with residual_pop>0)"
)

agent_scale = 20
agent_cap = 200

move_edges = []
for (u, v, data) in H_flow.edges(data=True):
    if data.get("kind") != "move":
        continue
    flow_people = flow_dict.get(u, {}).get(v, 0)
    if flow_people <= 0:
        continue
    ti = u[1]
    if ti >= len(time_names):
        continue
    geom = geom_lookup.get((int(u[0]), int(v[0]))) or geom_lookup.get((int(v[0]), int(u[0])))
    if geom is None:
        continue
    travel_hours = float(data.get("travel_hours", timestep_hour[min(ti, len(timestep_hour)-1)]))
    # Tag as exit-bound if the SOURCE node is on a flow path that drains to an exit
    is_exit_edge = u in exit_reachable_te
    move_edges.append((u, v, flow_people, geom, travel_hours, is_exit_edge))


def edge_to_agents_over_time(u, v, flow_people, geom, travel_hours, is_exit_edge):
    ti = u[1]
    n_agents = int(min(max(flow_people / agent_scale, 1), agent_cap))
    if n_agents <= 0:
        return []
    if travel_hours <= 0:
        travel_hours = 1e-6
    start_time = time_stamps[ti]
    end_time = start_time + pd.to_timedelta(travel_hours, unit="h")
    max_frame = len(time_names) - 1
    
    ts_array = time_stamps.to_numpy(dtype="datetime64[ns]")
    end_time_np = np.datetime64(end_time)

    arrival_frame = int(np.searchsorted(ts_array, end_time_np, side="left"))
    last_frame = min(max_frame, max(arrival_frame, ti))
    pts = []
    for frame in range(ti, last_frame + 1):
        frame_time = time_stamps[frame]
        elapsed = (frame_time - start_time).total_seconds() / 3600.0
        progress = min(max(elapsed / travel_hours, 0.0), 1.0)
        for j in range(n_agents):
            frac = (j + 1) / (n_agents + 1)
            pt = geom.interpolate(frac * progress, normalized=True)
            pts.append((frame, pt.x, pt.y, is_exit_edge))
    return pts


agent_chunks = Parallel(n_jobs=cfg.N_JOBS, prefer="threads")(
    delayed(edge_to_agents_over_time)(u, v, flow_people, geom, travel_hours, is_exit_edge)
    for (u, v, flow_people, geom, travel_hours, is_exit_edge) in move_edges
)

agent_positions_shelter  = [[] for _ in time_names]
agent_positions_out_city = [[] for _ in time_names]
for chunk in agent_chunks:
    for ti, x, y, is_exit in chunk:
        if is_exit:
            agent_positions_out_city[ti].append((x, y))
        else:
            agent_positions_shelter[ti].append((x, y))

n_shelter_dots  = sum(len(f) for f in agent_positions_shelter)
n_out_city_dots = sum(len(f) for f in agent_positions_out_city)
print(f"Animation dots — shelter-bound: {n_shelter_dots}  |  exit-bound: {n_out_city_dots}")
if n_shelter_dots + n_out_city_dots == 0:
    print("No evac movement to animate. Check flow_results and parameters.")

fig, ax = plt.subplots(figsize=(8, 8))


def update(frame_idx):
    ax.clear()
    time_name  = time_names[frame_idx]
    time_label = time_stamps[frame_idx]
    col = f"flood_depth_{time_name}"
    open_mask   = edges_anim[col].fillna(0) < cfg.IMPASSABLE_FLOOD_DEPTH
    closed_mask = ~open_mask

    open_edges  = edges_anim[open_mask]
    closed_edges = edges_anim[closed_mask]
    if len(open_edges):
        open_edges.plot(ax=ax, color="#4caf50", linewidth=0.6, alpha=0.8, label="Open edge", aspect="auto")
    if len(closed_edges):
        closed_edges.plot(ax=ax, color="#d32f2f", linewidth=0.6, alpha=0.5, label="Closed edge", aspect="auto")

    # Origin nodes colored by assigned destination
    if has_dest:
        for dest, color, lbl in [("shelter",  "#2196f3", "→ In-city shelter"),
                                   ("out_city", "#ff6f00", "→ Out-of-city exit")]:
            mask = nodes_anim["evac_dest"] == dest
            if mask.any():
                nodes_anim[mask].plot(ax=ax, color=color, markersize=pop_size[mask],
                                      alpha=0.7, edgecolor="k", linewidth=0.2, label=lbl)
    else:
        nodes_anim.plot(ax=ax, color="#2196f3", markersize=pop_size,
                        alpha=0.8, edgecolor="k", linewidth=0.2, label="Origins")

    # In-city shelter markers
    shelter_mask = nodes_anim["is_shelter"] == True
    if shelter_mask.any():
        nodes_anim[shelter_mask].plot(ax=ax, color="#1565c0", markersize=60, marker="^",
                                      edgecolor="k", linewidth=0.4, label="In-city shelter")

    # Out-of-city exit markers (orange ★)
    if len(out_city_anim) > 0:
        out_city_anim.plot(ax=ax, color="#ff6f00", markersize=180, marker="*",
                           edgecolor="k", linewidth=0.5, label="Out-of-city exit", zorder=6)

    # Stranded population (only once the node actually becomes unreachable)
    stranded_time = stranded_from_series
    stranded_mask = (
        nodes_anim["residual_pop"] > 0
    ) & stranded_time.notna() & (frame_idx >= stranded_time)
    stranded = nodes_anim[stranded_mask]
    if len(stranded):
        stranded.plot(ax=ax, color="#f44336", markersize=70, marker="x",
                      linewidth=1.2, label="Stranded")

    # Moving dots: blue = toward in-city shelter, orange = toward out-of-city exit
    if agent_positions_shelter[frame_idx]:
        xs, ys = zip(*agent_positions_shelter[frame_idx])
        ax.scatter(xs, ys, color="#1565c0", s=22, alpha=0.9, label="Flow → shelter")
    if agent_positions_out_city[frame_idx]:
        xs, ys = zip(*agent_positions_out_city[frame_idx])
        ax.scatter(xs, ys, color="#ff6f00", s=22, alpha=0.9, label="Flow → out-city")

    ax.set_title(f"Network status: {time_name} ({time_label})")
    ax.set_axis_off()
    if len(open_edges) or len(closed_edges):
        ax.set_aspect("equal", adjustable="datalim")
    else:
        ax.set_aspect("auto")
    handles, labels = ax.get_legend_handles_labels()
    uniq = dict(zip(labels, handles))
    if uniq:
        ax.legend(uniq.values(), uniq.keys(), loc="lower left", fontsize=7)
    return ax


ani = FuncAnimation(fig, update, frames=len(time_names), interval=800, repeat=True)


In [ ]:
# Saving animation to gif file
fps = 3
ani.save(f"{folder_path}/evac_animation_{fps}.gif", writer="pillow", fps=fps)
plt.show()

Visualizaing waiting edges congestion

In [ ]:
from collections import defaultdict

node_time_volume = defaultdict(float)

for u in flow_dict:
    for v, flow in flow_dict[u].items():
        if flow <= 0:
            continue

        # check if it's a waiting edge
        if isinstance(u, tuple) and isinstance(v, tuple):
            osmid_u, t_u = u
            osmid_v, t_v = v

            if osmid_u == osmid_v and t_v == t_u + 1:
                # this is a waiting edge
                node_time_volume[(osmid_u, t_u)] += flow

top_nodes = sorted(
    node_time_volume.items(),
    key=lambda x: x[1],
    reverse=True
)[:10]

for (osmid, t), vol in top_nodes:
    print(f"Node {osmid} at time {t}: {vol}")